# 🎤 Integrating Voice and Fingerprint Biometrics
## Age Estimation (LSTM) & Gender Detection (CNN)

**Instructions:**
1. Go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Run each cell in order.
3. Upload datasets when prompted or mount Google Drive.
4. After training, download `age_model.h5` and `gender_model.h5` from the file panel on the left.
5. Place them inside your local `backend/models/` folder.

## 1. Install Dependencies

In [ ]:
!pip install -q librosa opencv-python-headless pydub scikit-learn tensorflow pandas numpy matplotlib seaborn

## 2. Import Libraries

In [ ]:
import os
import cv2
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, LSTM, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

## 3. Mount Google Drive / Download Datasets
Uncomment the method you prefer.

In [ ]:
# ---- Option A: Mount Google Drive ----
# from google.colab import drive
# drive.mount('/content/drive')
# VOICE_CSV_PATH = '/content/drive/MyDrive/datasets/cv-valid-train.csv'
# VOICE_CLIPS_DIR = '/content/drive/MyDrive/datasets/cv-valid-train/'
# FINGERPRINT_DIR = '/content/drive/MyDrive/datasets/SOCOFing/Real/'

# ---- Option B: Download from Kaggle (requires API key) ----
# !pip install -q kaggle
# from google.colab import files
# files.upload()  # Upload your kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Common Voice dataset
# !kaggle datasets download -d mozillaorg/common-voice
# !unzip -q common-voice.zip -d /content/common_voice

# SOCOFing Fingerprint dataset
# !kaggle datasets download -d ruizgara/socofing
# !unzip -q socofing.zip -d /content/SOCOFing

# ---- Default paths (adjust as needed) ----
VOICE_CSV_PATH = '/content/cv_corpus_v1/cv-valid-train.csv'
VOICE_CLIPS_DIR = '/content/cv_corpus_v1/cv-valid-train/cv-valid-train/'
FINGERPRINT_DIR = '/content/SOCOFing/Real/'

print('Dataset paths configured.')

---
# 🖼️ Part A: Fingerprint Gender Detection (CNN)
---

### A1. Fingerprint Preprocessing

In [ ]:
def preprocess_fingerprint(image_path):
    """Load, enhance and normalize a fingerprint image to 96x96 grayscale."""
    try:
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        img = cv2.equalizeHist(img)
        img = cv2.GaussianBlur(img, (5, 5), 0)
        img = cv2.resize(img, (96, 96))
        img = img.astype('float32') / 255.0
        return np.expand_dims(img, axis=-1)
    except Exception as e:
        return None

print('Fingerprint preprocessing function defined.')

### A2. Load Fingerprint Data

In [ ]:
def load_fingerprint_data(dataset_dir, max_samples=5000):
    """Load SOCOFing fingerprint images, extract gender from filename."""
    X, y = [], []
    if not os.path.exists(dataset_dir):
        print(f'ERROR: Directory not found: {dataset_dir}')
        return np.array([]), np.array([])

    files = [f for f in os.listdir(dataset_dir) if f.endswith(('.BMP', '.bmp', '.png', '.jpg'))]
    np.random.shuffle(files)

    count = 0
    for filename in files:
        if count >= max_samples:
            break
        if '__M_' in filename or '_M_' in filename:
            gender_label = 0  # Male
        elif '__F_' in filename or '_F_' in filename:
            gender_label = 1  # Female
        else:
            continue

        features = preprocess_fingerprint(os.path.join(dataset_dir, filename))
        if features is not None:
            X.append(features)
            y.append(gender_label)
            count += 1

    print(f'Loaded {len(X)} fingerprint samples.')
    return np.array(X), np.array(y)

X_fp, y_gender = load_fingerprint_data(FINGERPRINT_DIR, max_samples=5000)
if len(X_fp) > 0:
    print(f'Shape: {X_fp.shape} | Male: {np.sum(y_gender==0)} | Female: {np.sum(y_gender==1)}')

### A3. Build & Train CNN

In [ ]:
if len(X_fp) > 0:
    X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
        X_fp, y_gender, test_size=0.2, random_state=42, stratify=y_gender
    )

    gender_model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(96, 96, 1)),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Conv2D(128, (3,3), activation='relu'),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    gender_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    gender_model.summary()

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
    ]

    print('\nTraining Gender CNN...')
    gender_history = gender_model.fit(
        X_train_f, y_train_f,
        epochs=50, batch_size=32,
        validation_split=0.15,
        callbacks=callbacks
    )
else:
    print('No fingerprint data loaded. Please check the dataset path.')

### A4. Evaluate & Save Gender Model

In [ ]:
if len(X_fp) > 0:
    loss, acc = gender_model.evaluate(X_test_f, y_test_f)
    print(f'\nTest Accuracy: {acc*100:.2f}%')

    # Confusion Matrix
    y_pred_f = (gender_model.predict(X_test_f) > 0.5).astype(int).flatten()
    cm = confusion_matrix(y_test_f, y_pred_f)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Male','Female'], yticklabels=['Male','Female'])
    plt.title('Gender CNN - Confusion Matrix')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig('gender_confusion_matrix.png', dpi=150)
    plt.show()

    print(classification_report(y_test_f, y_pred_f, target_names=['Male','Female']))

    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(gender_history.history['accuracy'], label='Train')
    axes[0].plot(gender_history.history['val_accuracy'], label='Val')
    axes[0].set_title('Gender CNN Accuracy'); axes[0].legend()
    axes[1].plot(gender_history.history['loss'], label='Train')
    axes[1].plot(gender_history.history['val_loss'], label='Val')
    axes[1].set_title('Gender CNN Loss'); axes[1].legend()
    plt.tight_layout()
    plt.savefig('gender_training_curves.png', dpi=150)
    plt.show()

    gender_model.save('gender_model.h5')
    print('Saved gender_model.h5')

---
# 🎤 Part B: Voice Age Estimation (LSTM)
---

### B1. Audio Preprocessing

In [ ]:
SEQUENCE_LENGTH = 160
N_MFCC = 64

def preprocess_audio(file_path):
    """Load audio, trim silence, extract MFCC sequence -> (160, 64)"""
    try:
        y, sr = librosa.load(file_path, sr=16000, duration=4.0)
        y_trimmed, _ = librosa.effects.trim(y, top_db=20)
        target_len = 4 * 16000
        if len(y_trimmed) < target_len:
            y_trimmed = np.pad(y_trimmed, (0, target_len - len(y_trimmed)))
        else:
            y_trimmed = y_trimmed[:target_len]
        mfcc = librosa.feature.mfcc(y=y_trimmed, sr=sr, n_mfcc=N_MFCC)
        mfcc_t = mfcc.T  # (timesteps, 64)
        if len(mfcc_t) < SEQUENCE_LENGTH:
            mfcc_t = np.pad(mfcc_t, ((0, SEQUENCE_LENGTH - len(mfcc_t)), (0, 0)))
        else:
            mfcc_t = mfcc_t[:SEQUENCE_LENGTH]
        return mfcc_t
    except Exception as e:
        return None

print('Audio preprocessing function defined.')

### B2. Load Voice Data

In [ ]:
AGE_MAP = {
    'teens': 15, 'twenties': 25, 'thirties': 35,
    'fourties': 45, 'fifties': 55, 'sixties': 65,
    'seventies': 75, 'eighties': 85, 'nineties': 95
}

def load_voice_data(csv_path, clips_dir, max_samples=3000):
    """Load Common Voice audio samples with valid age labels."""
    X, y = [], []
    if not os.path.exists(csv_path):
        print(f'ERROR: CSV not found: {csv_path}')
        return np.array([]), np.array([])

    df = pd.read_csv(csv_path)
    df = df.dropna(subset=['age'])
    df = df[df['age'].isin(AGE_MAP.keys())]
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    count = 0
    for _, row in df.iterrows():
        if count >= max_samples:
            break
        fpath = os.path.join(clips_dir, row['filename'])
        if not os.path.exists(fpath):
            continue
        features = preprocess_audio(fpath)
        if features is not None:
            X.append(features)
            y.append(AGE_MAP[row['age']])
            count += 1
            if count % 500 == 0:
                print(f'  Processed {count} samples...')

    print(f'Loaded {len(X)} voice samples.')
    return np.array(X), np.array(y)

X_voice, y_age = load_voice_data(VOICE_CSV_PATH, VOICE_CLIPS_DIR, max_samples=3000)
if len(X_voice) > 0:
    print(f'Shape: {X_voice.shape}')

### B3. Build & Train LSTM

In [ ]:
if len(X_voice) > 0:
    X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(
        X_voice, y_age, test_size=0.2, random_state=42
    )

    age_model = Sequential([
        LSTM(128, return_sequences=True, input_shape=(SEQUENCE_LENGTH, N_MFCC)),
        Dropout(0.3),
        LSTM(64),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='linear')
    ])
    age_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    age_model.summary()

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
    ]

    print('\nTraining Age LSTM...')
    age_history = age_model.fit(
        X_train_v, y_train_v,
        epochs=50, batch_size=32,
        validation_split=0.15,
        callbacks=callbacks
    )
else:
    print('No voice data loaded. Please check the dataset path.')

### B4. Evaluate & Save Age Model

In [ ]:
if len(X_voice) > 0:
    loss, mae = age_model.evaluate(X_test_v, y_test_v)
    print(f'\nTest MAE: {mae:.2f} years')

    y_pred_v = age_model.predict(X_test_v).flatten()

    # Scatter plot
    plt.figure(figsize=(8, 6))
    plt.scatter(y_test_v, y_pred_v, alpha=0.5, s=10)
    plt.plot([10, 100], [10, 100], 'r--')
    plt.xlabel('Actual Age'); plt.ylabel('Predicted Age')
    plt.title(f'Age LSTM - Actual vs Predicted (MAE={mae:.2f})')
    plt.tight_layout()
    plt.savefig('age_scatter.png', dpi=150)
    plt.show()

    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(age_history.history['mae'], label='Train')
    axes[0].plot(age_history.history['val_mae'], label='Val')
    axes[0].set_title('Age LSTM MAE'); axes[0].legend()
    axes[1].plot(age_history.history['loss'], label='Train')
    axes[1].plot(age_history.history['val_loss'], label='Val')
    axes[1].set_title('Age LSTM Loss'); axes[1].legend()
    plt.tight_layout()
    plt.savefig('age_training_curves.png', dpi=150)
    plt.show()

    age_model.save('age_model.h5')
    print('Saved age_model.h5')

---
## ✅ Download Models
Run the cell below to download both `.h5` files. Place them inside `backend/models/` in your local project.

In [ ]:
from google.colab import files
for f in ['gender_model.h5', 'age_model.h5']:
    if os.path.exists(f):
        files.download(f)
        print(f'Downloaded {f}')
    else:
        print(f'{f} not found. Train that model first.')